In [ ]:
import numpy as np, pandas as pd, joblib, warnings, xgboost as xgb
from pathlib import Path
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (mean_squared_log_error, mean_squared_error,
                             mean_absolute_error, r2_score)
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
warnings.filterwarnings("ignore")

In [ ]:
# -----------------------------
# Yardımcı fonksiyonlar
# -----------------------------
def rmsle(y_true, y_pred):
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

def rare_zip(series, top_n=20):
    top = series.value_counts().nlargest(top_n).index
    return series.where(series.isin(top), "other").astype(str)

def feature_engineer(df: pd.DataFrame) -> pd.DataFrame:
    """Ham veriden öznitelik mühendisliğini uygular."""
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["year_sold"]  = df["date"].dt.year
    df["month_sold"] = df["date"].dt.month
    df["age"]        = df["year_sold"] - df["yr_built"]
    df["renov_age"]  = np.where(df["yr_renovated"]==0, 0,
                                df["year_sold"] - df["yr_renovated"])
    df["bath_per_bed"]     = df["bathrooms"] / df["bedrooms"].replace(0, np.nan)
    df["living_lot_ratio"] = df["sqft_living"] / df["sqft_lot"].replace(0, np.nan)
    df["zipcode"]          = rare_zip(df["zipcode"])
    df.drop(columns=["id", "date"], inplace=True)
    return df

def evaluate(y_true, y_pred):
    return {
      "RMSLE": rmsle(y_true, y_pred),
      "RMSE":  mean_squared_error(y_true, y_pred, squared=False),
      "MAE":   mean_absolute_error(y_true, y_pred),
      "R2":    r2_score(y_true, y_pred)
    }

In [ ]:
# -----------------------------
# 1) Veri yükle & ön-işleme
# -----------------------------
df_raw = pd.read_csv(Path("/kaggle/input/house-data/house_data.csv"))
df     = feature_engineer(df_raw)

y = np.log1p(df["price"])          # hedef log1p
X = df.drop(columns=["price"])

cat_cols = ["zipcode", "view", "condition", "grade"]
num_cols = [c for c in X.columns if c not in cat_cols]

# Ortak önişleme (Lineer, RF, GB, SVR)
prep_common = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# Polinom için sayısallara 2. derece, sonrasında Ridge
poly_tf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly",    PolynomialFeatures(degree=2, include_bias=False))
])
prep_poly = ColumnTransformer([
    ("poly_num", poly_tf, num_cols),
    ("cat",      OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [ ]:
# -----------------------------
# 2) Model havuzu
# -----------------------------
models = {
    "Linear": Pipeline([
        ("prep", prep_common),
        ("reg",  LinearRegression())
    ]),
    "Poly_Ridge": Pipeline([
        ("prep", prep_poly),
        ("reg",  Ridge(alpha=1.0, random_state=42))
    ]),
    "RandomForest": Pipeline([
        ("prep", prep_common),
        ("reg",  RandomForestRegressor(
                    n_estimators=600,
                    max_depth=None,
                    min_samples_leaf=2,
                    n_jobs=-1,
                    random_state=42))
    ]),
    "GradientBoost": Pipeline([
        ("prep", prep_common),
        ("reg",  GradientBoostingRegressor(
                    n_estimators=800,
                    learning_rate=0.05,
                    max_depth=3,
                    random_state=42))
    ]),
    "SVR": Pipeline([
        ("prep", prep_common),
        ("reg",  SVR(kernel="rbf", C=200, gamma=0.03, epsilon=0.05))
    ])
}

In [ ]:
# -----------------------------
# 3) Eğitim / Değerlendirme
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

metric_table = {}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    metric_table[name] = evaluate(y_test, y_pred)
    print(f"{name:12s}  RMSLE={metric_table[name]['RMSLE']:.4f}  "
          f"RMSE={metric_table[name]['RMSE']:.0f}  "
          f"MAE={metric_table[name]['MAE']:.0f}  "
          f"R²={metric_table[name]['R2']:.3f}")

In [ ]:
# -----------------------------
# 4) En iyi modeli seç ve kaydet
#    (kriter: en düşük RMSLE)
# -----------------------------
best_name = min(metric_table, key=lambda k: metric_table[k]["RMSLE"])
best_model = models[best_name]
print("\nEn iyi model:", best_name)